In [1]:
import pandas as pd
import numpy as np

customers = pd.read_csv("./data/raw/customers.csv")
orders = pd.read_csv("./data/raw/orders.csv")
order_items = pd.read_csv("./data/raw/order_items.csv")
products = pd.read_csv("./data/raw/products.csv")

print(customers.head())
print(orders.head())
print(order_items.head())
print(products.head())

   customer_id name gender  age city signup_date
0            1  김수민      F   19   광주  2024-08-22
1            2  김정호      F   32   대구  2026-01-05
2            3  이경수      F   61   성남  2024-08-15
3            4  조영호      F   55   울산  2026-06-16
4            5  이예원      F   19   부산  2024-11-16
   order_id  customer_id  order_date payment_method order_status
0         1          123  2026-07-10           card    completed
1         2           77  2025-09-25      naver_pay    cancelled
2         3          138  2026-01-22  bank_transfer    cancelled
3         4           57  2026-04-04      kakao_pay    cancelled
4         5          125  2026-02-23           card    cancelled
   order_item_id  order_id  product_id  quantity  unit_price
0              1         1         100         3      102000
1              2         1          87         5       25000
2              3         1           7         3      142000
3              4         1           9         3      193000
4          

In [2]:
# 원본 구조 Evidence를 만든다.

raw_data = {"customers" : customers, "order_items" : order_items, "products": products, "orders" : orders}

print(raw_data)

{'customers':      customer_id name gender  age city signup_date
0              1  김수민      F   19   광주  2024-08-22
1              2  김정호      F   32   대구  2026-01-05
2              3  이경수      F   61   성남  2024-08-15
3              4  조영호      F   55   울산  2026-06-16
4              5  이예원      F   19   부산  2024-11-16
..           ...  ...    ...  ...  ...         ...
145          146  김숙자      M   61   성남  2026-02-25
146          147  이정남      M   19   부산  2025-04-16
147          148  오도현      M   29   고양  2026-08-18
148          149  김정자      M   20   부산  2024-12-22
149          150  조미영      M   40   대전  2026-02-06

[150 rows x 6 columns], 'order_items':      order_item_id  order_id  product_id  quantity  unit_price
0                1         1         100         3      102000
1                2         1          87         5       25000
2                3         1           7         3      142000
3                4         1           9         3      193000
4                5 

In [3]:
for name, frame in raw_data.items():
   print(name)

customers
order_items
products
orders


In [4]:
summary_list = []

for name, frame in raw_data.items():
    info = {
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "missing_values": int(frame.isna().sum().sum()),
        "duplicated_rows": int(frame.duplicated().sum()),
    }
    summary_list.append(info)

summary = pd.DataFrame(summary_list)
print(summary)

       dataset  rows  columns  missing_values  duplicated_rows
0    customers   150        6               0                0
1  order_items   764        5               0                0
2     products   100        4               0                0
3       orders   300        5               0                0


In [5]:
from src.preprocessing import (

    compare_shapes,

    preprocess_sales_data,

    validate_relationships,

)

processed_data = preprocess_sales_data(raw_data)

preprocessing_comparison = compare_shapes(

    raw_data,

    processed_data,

)

relationship_checks = validate_relationships(

    processed_data

)

In [6]:
from src.preprocessing import compare_shapes, preprocess_sales_data

processed_data = preprocess_sales_data(raw_data)
preprocessing_comparison = compare_shapes(raw_data, processed_data)

In [7]:
preprocessing_comparison

,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,150,6,150,6
1,order_items,764,5,764,6
2,orders,300,5,300,7
3,products,100,4,100,4


In [12]:
print(raw_data["order_items"].columns)
print(processed_data["order_items"].columns)

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price'], dtype='str')
Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'line_total'],
      dtype='str')


In [11]:
print(processed_data["orders"].head())

   order_id  customer_id order_date payment_method order_status order_month  \
0         1          123 2026-07-10           card    completed     2026-07   
1         2           77 2025-09-25      naver_pay    cancelled     2025-09   
2         3          138 2026-01-22  bank_transfer    cancelled     2026-01   
3         4           57 2026-04-04      kakao_pay    cancelled     2026-04   
4         5          125 2026-02-23           card    cancelled     2026-02   

  order_dayofweek  
0          Friday  
1        Thursday  
2        Thursday  
3        Saturday  
4          Monday  


In [16]:
print(processed_data["orders"]["order_dayofweek"].value_counts)

<bound method IndexOpsMixin.value_counts of 0        Friday
1      Thursday
2      Thursday
3      Saturday
4        Monday
         ...   
295     Tuesday
296      Friday
297      Friday
298    Saturday
299    Thursday
Name: order_dayofweek, Length: 300, dtype: str>


key값 체크

In [17]:
key_map = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
}

In [ ]:
pk_checks = []

for dataset, key in key_map.items():
    frame = processed_data[dataset]

    missing_count = int(frame[key].isna().sum())
    duplicated_count = int(frame[key].duplicated().sum())

    if missing_count == 0 and duplicated_count == 0:
        status = "PASS"
    else:
        status = "FAIL"

    pk_checks.append(
        {
            "dataset": dataset,
            "key": key,
            "missing_count": missing_count,
            "duplicated_count": duplicated_count,
            "status": status,
        }
    )

pd.DataFrame(pk_checks)


In [ ]:
# null값이 있는지 

